# 1) Imports and Global Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

DATASET_ID = 'FD001' # Flip this between 'FD001' and 'FD002'
WINDOW_SIZE = 30
RUL_CAP = 125

# 2) Loading Raw Data Dynamically

In [ ]:
# Standardised C-MAPSS column mapping
column_names = (
    ['unit', 'cycle']
    + [f'op_setting_{i}' for i in range(1, 4)]
    + [f'sensor_{i}' for i in range(1, 22)]
)

raw_path = f'../data/raw/train_{DATASET_ID}.txt'
print(f"Loading raw dataset from: {raw_path}")

train_df = pd.read_csv(
    raw_path,
    sep=r'\s+',
    header=None,
    names=column_names
)

train_df.head()

# 3) Target Engineering (Piecewise RUL)

In [ ]:
# Calculation absolute remaining useful life
rul = train_df.groupby('unit')['cycle'].max().reset_index()
rul.columns = ['unit', 'max_cycle']

train_df = train_df.merge(rul, on='unit', how='left')
train_df['RUL'] = train_df['max_cycle'] - train_df['cycle']

# Apply standard piece-wise linear RUL bounding
train_df['RUL_capped'] = train_df['RUL'].clip(upper=RUL_CAP)

print(f"Dataset shape after target engineering: {train_df.shape}")

# 4) Dynamic Feature Selection (Zero-Variance Drop)

In [ ]:
# Identify non-predictive features where global max equals global min
meta_cols = ['unit', 'cycle', 'max_cycle', 'RUL', 'RUL_capped']
constant_cols = [
    col for col in train_df.columns 
    if col not in meta_cols and train_df[col].min() == train_df[col].max()
]

print(f"Dynamically dropping zero-variance features: {constant_cols}")
train_df.drop(columns=constant_cols, inplace=True)

# Define remaining predictive features
feature_cols = [col for col in train_df.columns if col not in meta_cols]
print(f"Total features retained for processing ({len(feature_cols)}): {feature_cols}")

# 5) Hybrid Regime Clustering & Scaling

In [ ]:
import pickle
import os

os.makedirs('../scalers', exist_ok=True)

# First, Isolate available operational conditions vs physical engine sensors
op_settings_available = [col for col in ['op_setting_1', 'op_setting_2', 'op_setting_3'] if col in train_df.columns]
sensor_cols = [col for col in feature_cols if col not in op_settings_available]

# Second, Build cluster tags using nominal altitude integers to avoid rounding boundary splits
regime_clusters = train_df['op_setting_1'].round(0).astype(int).astype(str)

print(f"\nIdentified {regime_clusters.nunique()} Operational Regime Clusters for {DATASET_ID}:")
print(regime_clusters.value_counts())

# Third, Local Scale SENSORS (Eliminates environmental shift across flight conditions)
scaled_group_list = []
fitted_scalers = {}  # Dictionary to collect scalers for each regime

for regime, group in train_df.groupby(regime_clusters):
    group_scaled = group.copy()
    local_scaler = MinMaxScaler()
    
    # Fit and transform the training sensor data
    group_scaled[sensor_cols] = local_scaler.fit_transform(group[sensor_cols])
    scaled_group_list.append(group_scaled)
    
    # Capture the fitted scaler state mapped to this specific regime string
    fitted_scalers[str(regime)] = local_scaler

# Serialise the complete dictionary of training scalers to disk
scaler_path = f'../scalers/train_scalers_{DATASET_ID}.pkl'
with open(scaler_path, 'wb') as f:
    pickle.dump(fitted_scalers, f)

print(f"Training scalers locked down and saved successfully to {scaler_path}!")

recombined_df = pd.concat(scaled_group_list).sort_index()

# Fourth, Global Scale OPERATIONAL SETTINGS (Retains clean absolute context for LSTM)
if op_settings_available:
    global_op_scaler = MinMaxScaler()
    recombined_df[op_settings_available] = global_op_scaler.fit_transform(recombined_df[op_settings_available])

# Fifth, Build final standardized master dataframe
scaled_df = recombined_df[feature_cols].copy()
scaled_df['unit'] = train_df['unit'].values
scaled_df['cycle'] = train_df['cycle'].values
scaled_df['RUL'] = train_df['RUL_capped'].values

print("\nFeatures scaled and standardized successfully.")
scaled_df.head()

# 6) Engine-Based Stratified Train/Validation Split

In [ ]:
unique_units = scaled_df['unit'].unique()

# Split execution by physical engine ID to eliminate validation leakage
train_units, val_units = train_test_split(
    unique_units,
    test_size=0.2,
    random_state=42,
    shuffle=True
)

train_scaled_df = scaled_df[scaled_df['unit'].isin(train_units)].copy()
val_scaled_df = scaled_df[scaled_df['unit'].isin(val_units)].copy()

print(f"Training subset: {len(train_units)} engines ({len(train_scaled_df)} rows)")
print(f"Validation subset: {len(val_units)} engines ({len(val_scaled_df)} rows)")

# 7) Sliding Window Sequence Generation

In [ ]:
def create_sequences(df, feature_cols, window_size):
    X_seq, y_seq = [], []
    
    for unit in df['unit'].unique():
        unit_df = df[df['unit'] == unit].sort_values('cycle')
        
        # Enforce sequence collection boundary
        for i in range(len(unit_df) - window_size):
            window = unit_df.iloc[i : i + window_size][feature_cols].values
            target = unit_df.iloc[i + window_size]['RUL']
            
            X_seq.append(window)
            y_seq.append(target)
            
    return np.array(X_seq), np.array(y_seq)

# Build sequential arrays
X_train_seq, y_train_seq = create_sequences(train_scaled_df, feature_cols, WINDOW_SIZE)
X_val_seq, y_val_seq = create_sequences(val_scaled_df, feature_cols, WINDOW_SIZE)

print(f"\nFinal Windowed Sequence Shapes for {DATASET_ID}:")
print(f"X_train_seq: {X_train_seq.shape}")
print(f"y_train_seq: {y_train_seq.shape}")
print(f"X_val_seq:   {X_val_seq.shape}")
print(f"y_val_seq:   {y_val_seq.shape}")

# 8) Secure Array Serialization

In [ ]:
# Dynamic path generation completely avoids cross-dataset overwriting
x_train_out = f'../data/processed/X_train_{DATASET_ID}_engine_split.npy'
y_train_out = f'../data/processed/y_train_{DATASET_ID}_engine_split.npy'
x_val_out   = f'../data/processed/X_val_{DATASET_ID}_engine_split.npy'
y_val_out   = f'../data/processed/y_val_{DATASET_ID}_engine_split.npy'

np.save(x_train_out, X_train_seq)
np.save(y_train_out, y_train_seq)
np.save(x_val_out, X_val_seq)
np.save(y_val_out, y_val_seq)

print("Sequences successfully locked down to disk!")